# La régression sans les anomalies

La détection d'anomalies du projet tourne d'abord. Les salariés qu'elle classe dans les priorités choisies (`Critical` pour commencer) sortent du tableau. Puis la régression de `Prediction_Compa_Ratio` est refaite sur ceux qui restent, avec les mêmes étapes : les corrélations, les six modèles. À la fin, une validation croisée.

Avec `A_RETIRER = []` à l'étape 2, personne n'est retiré : la régression se fait sur tous les salariés, et on retrouve les résultats de `Prediction_Compa_Ratio` (avec son réglage `ENLEVER = False`).

Tout ce que le notebook enregistre s'écrit dans son dossier, à côté de lui.

---
## 1. Lancer la détection d'anomalies

Les fonctions sont celles du projet (dossier `src/`), avec les réglages de `config/rules.yaml`. Chaque salarié reçoit un score et une priorité : `Critical`, `Major`, `Minor` ou `Info`.

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import yaml
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Les fonctions de la détection sont celles du projet, dans src/ : elles ne sont pas recopiées ici
sys.path.append('../src')
from anomaly_pretraitement import preparer_employes
from anomaly_cohortes import former_cohortes
from anomaly_regles import apply_rulebook
from anomaly_signal_iforest import ml_anomaly
from anomaly_signal_fort import apply_ml_strong_signal
from anomaly_score_general import aggregate_risk
from anomaly_recommandations import recommendations

DOSSIER = '../input'   # le dossier des trois fichiers Excel
px.defaults.template = 'plotly_white'   # des graphes sur fond blanc
pd.set_option('display.precision', 2)   # les tableaux affichent 2 chiffres après la virgule ; les calculs gardent toute leur précision

# Les réglages de la détection (seuils, poids, priorités) : ceux du projet
with open('../config/rules.yaml', encoding='utf-8') as fichier:
    regles = yaml.safe_load(fichier)['rules']

# Les étapes de la détection, dans le même ordre que le programme principal (src/detect_salary_anomalies.py)
detection = preparer_employes(f'{DOSSIER}/employes.xlsx', f'{DOSSIER}/bands.xlsx', f'{DOSSIER}/market.xlsx')
for etape in (former_cohortes, apply_rulebook, ml_anomaly, apply_ml_strong_signal, aggregate_risk, recommendations):
    detection = etape(detection, regles)

# Le résultat complet de la détection, enregistré à côté du notebook
detection.round(2).to_excel('anomalies.xlsx', index=False)
print(f"{len(detection)} salariés analysés. Résultat de la détection enregistré dans anomalies.xlsx")

# Combien de salariés dans chaque priorité, de la plus grave à la moins grave
PRIORITES = list(regles['prioritization_buckets'])[::-1]
display(detection['Severity'].value_counts().reindex(PRIORITES).fillna(0).astype(int).to_frame('Salariés'))

---
## 2. Choisir les anomalies à retirer

`A_RETIRER` est la liste des priorités qu'on retire. Pour en retirer plus, ajouter une priorité dans la liste, puis relancer à partir d'ici.

Avec une liste vide `[]`, personne n'est retiré : la régression se fait sur tous les salariés.

In [ ]:
# [] : personne n'est retiré, la régression se fait sur tous les salariés.
# Pour en retirer plus : ['Critical', 'Major'], ou ['Critical', 'Major', 'Minor']
A_RETIRER = ['Critical']

inconnues = [p for p in A_RETIRER if p not in PRIORITES]
if inconnues:
    print(f"Priorité inconnue, ignorée : {inconnues}. Les priorités possibles : {PRIORITES}")

retires = detection[detection['Severity'].isin(A_RETIRER)]
print(f"{len(retires)} salariés à retirer sur {len(detection)} ({len(retires) / len(detection):.1%}).")
if retires.empty:
    print("Aucun salarié n'a cette priorité : personne n'est retiré, et la suite se fait sur tout le monde.")

# La liste des salariés retirés, avec tout ce que la détection dit d'eux, enregistrée à côté du notebook
retires.round(2).to_excel('salaries_retires.xlsx', index=False)
print("Liste enregistrée dans salaries_retires.xlsx")
a_montrer = ['Matricule', 'Job_Family', 'Niveau', 'Fixe_Annuel_MAD', 'Min', 'Mid', 'RiskScore', 'Severity', 'Reason_Principale']
display(retires[[c for c in a_montrer if c in retires.columns]].sort_values('RiskScore', ascending=False).head(10))

---
## 3. Le tableau de la régression

Le tableau sorti de la détection sert directement à la régression : chaque salarié y a déjà sa grille et son marché. Les ratios sont calculés ici, avec les mêmes formules que dans `Prediction_Compa_Ratio`. Puis les anomalies choisies à l'étape 2 sortent du tableau : toute la suite se fait sans elles.

Le tableau garde toutes les colonnes. Le choix de celles qui servent au modèle se fait à l'étape 4, juste avant les corrélations.

In [ ]:
CIBLES = ['Fixe_Annuel_MAD', 'Compa_Ratio']   # les deux variables à prédire : le salaire fixe et le Compa-Ratio

# 1. On part du tableau de la détection : chaque salarié y a déjà sa grille (Min, Mid, Max)
#    et la médiane de son marché (Median).
df = detection.copy()

# 2. Les ratios, calculés ici avec les mêmes formules que dans Prediction_Compa_Ratio
df['Compa_Ratio'] = df['Fixe_Annuel_MAD'] / df['Mid']
df['Ratio_Grille_Marche'] = df['Mid'] / df['Median']
if 'Age' in df.columns and 'Anciennete' in df.columns:
    df['Age_Embauche'] = df['Age'] - df['Anciennete']
# Une division par zéro donne « infini » : on remet la case à vide
df = df.replace([np.inf, -np.inf], np.nan)
# Un ratio négatif ou nul est impossible (salaire à 0 ou négatif) : case vidée aussi
for c in ['Compa_Ratio', 'Ratio_Grille_Marche']:
    df.loc[df[c] <= 0, c] = np.nan

# 3. Un salarié sans Compa-Ratio (pas de grille, ou pas de salaire) ne peut pas servir : on le retire
sans_cible = df['Compa_Ratio'].isna()
df = df[~sans_cible].reset_index(drop=True)
print(f"{len(df)} salariés gardés, {int(sans_cible.sum())} retirés (pas de grille ou pas de salaire).")

# 4. Une colonne vide, ou qui a la même valeur pour tout le monde, ne peut rien expliquer : on la retire.
#    Les nombres à virgule sont arrondis au millième : un écart d'arrondi (0,6600001) n'est pas une vraie différence.
inutiles = [c for c in df.columns if (df[c].round(3) if pd.api.types.is_float_dtype(df[c]) else df[c]).nunique() <= 1]
df = df.drop(columns=inutiles)
if inutiles:
    print(f"Colonnes vides ou identiques pour tous, retirées : {inutiles}")

# 5. Les anomalies choisies à l'étape 2 sortent du tableau, repérées par leur matricule.
#    Toute la suite (corrélations, modèles) se fait sans elles.
anomalie = df['Matricule'].isin(retires['Matricule'])
df = df[~anomalie]
print(f"{int(anomalie.sum())} anomalies retirées du tableau, {len(df)} salariés pour la suite.")

---
## 4. Les corrélations

C'est ici qu'on choisit les colonnes du modèle, en trois temps :

1. **la liste** des colonnes candidates, juste en dessous ;
2. **la matrice et le tableau** : le lien de chaque colonne avec ce qu'on veut prédire, et les colonnes qui se répètent ;
3. **ce que chaque colonne apporte** au modèle, quand on la retire.

Quand la liste convient, on passe aux modèles.

In [ ]:
COLONNES = [
    'Niveau',                  # le niveau du poste
    'Niveau_Etude_Max',        # le niveau d'études
    'Age_Embauche',            # âge − ancienneté
    'Anciennete',              # l'ancienneté, en années
    'Competence_N1',           # la note de compétence de l'an dernier
    'Performance_N1',          # la performance de l'an dernier
    'Hot_job',                 # la tension du métier sur le marché
    'Ratio_Grille_Marche',     # Mid / médiane du marché
]

# L'ancienneté en buckets. True : la colonne Bucket_Anciennete s'ajoute aux colonnes utilisées
# (matrice, tableau, modèle). False : elle n'existe nulle part.
BUCKETS_ANCIENNETE = False
# Les bornes des buckets, en années : 1 = de 0 à 2 ans, 2 = de 2 à 5 ans, 3 = de 5 à 10 ans, puis un bucket
# tous les 5 ans : 4 = de 10 à 15 ans, ... 9 = 35 ans et plus. Pour d'autres buckets, changer les bornes.
BORNES_ANCIENNETE = [2, 5, 10, 15, 20, 25, 30, 35]

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. On part du tableau complet (df) et on ne garde que les deux cibles et les colonnes de la liste.
#    df lui-même n'est pas modifié : on peut changer la liste et relancer à partir de la cellule du dessus.
absentes = [c for c in COLONNES if c not in df.columns]
if absentes:
    print(f"Colonnes de la liste ignorées (absentes du fichier, ou identiques pour tous les salariés) : {absentes}")
if [c for c in COLONNES if c in CIBLES]:
    print(f"Ignoré : {[c for c in COLONNES if c in CIBLES]}. Ce sont les variables à prédire : ce serait donner la réponse aux modèles.")
donnees = df[CIBLES + [c for c in COLONNES if c in df.columns and c not in CIBLES]].copy()

# 2. L'ancienneté en buckets : une seule colonne, le numéro du bucket (1 = le moins d'ancienneté)
if BUCKETS_ANCIENNETE and 'Anciennete' in df.columns:
    donnees['Bucket_Anciennete'] = pd.cut(df['Anciennete'], [-np.inf] + BORNES_ANCIENNETE + [np.inf], labels=False, right=False) + 1
    print(f"Salariés par bucket d'ancienneté : {donnees['Bucket_Anciennete'].dropna().astype(int).value_counts().sort_index().to_dict()}")
elif BUCKETS_ANCIENNETE:
    print("Pas de buckets d'ancienneté : le fichier n'a pas de colonne Anciennete.")

# 3. La performance et le niveau d'études passent en rang, pour apparaître dans la matrice.
# Un libellé inconnu compte comme une case vide : il prend le rang médian.
# Si la plupart des libellés du fichier ne sont pas ceux prévus, on ne force rien : la colonne reste
# en texte et n'est pas utilisée.
RANG_PERFORMANCE = {"Très en dessous": 1, "Proche de l'objectif": 2, "A l'objectif": 3, "Supérieur à l'objectif": 4, "Très au-delà": 5}
RANG_ETUDE = {'SECONDAIRE': 0, 'BAC': 1, 'BAC+1': 2, 'BAC+2': 3, 'BAC+3': 4, 'BAC+4': 5, 'BAC+5': 6, 'BAC+6': 6,
              'BAC+7': 7, 'EXPERTISE COMPTABLE': 7}
for colonne, rangs in [('Performance_N1', RANG_PERFORMANCE), ('Niveau_Etude_Max', RANG_ETUDE)]:
    if colonne in donnees.columns and donnees[colonne].isin(list(rangs)).mean() > 0.5:
        donnees[colonne] = donnees[colonne].map(rangs)
        donnees[colonne] = donnees[colonne].fillna(donnees[colonne].median())

# 4. Les colonnes restées en texte (le métier, par exemple) ne sont pas utilisées :
#    les modèles ne lisent que des chiffres.
en_texte = list(donnees.select_dtypes(exclude='number').columns)
if en_texte:
    print(f"Colonnes en texte non utilisées : {en_texte}")
    donnees = donnees.drop(columns=en_texte)

# 5. Un salarié qui a une case vide dans une des colonnes chiffrées choisies sort : le modèle ne sait pas lire une case vide
vides = donnees.select_dtypes('number').isna().any(axis=1)
donnees = donnees[~vides]
print(f"{len(donnees)} salariés pour la suite ({int(vides.sum())} retirés pour une case vide dans une des colonnes choisies).")

# 6. La matrice des colonnes chiffrées. Les deux cibles sont sur les deux premières lignes
chiffrees = donnees.select_dtypes('number')
corr = chiffrees.corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, height=700,
                title='Corrélations des colonnes chiffrées')
fig.show()


def liens_et_vif(tableau):
    # Pour chaque colonne chiffrée : son lien avec chacune des deux cibles, et son VIF.
    # Le lien est sa corrélation avec la cible, entre −1 et +1 (les deux premières lignes de la matrice).
    # Le VIF ne mesure pas l'importance : il dit si la colonne est déjà dite par les AUTRES colonnes chiffrées.
    # 1 = pas du tout ; au-dessus de 5 = à regarder ; au-dessus de 10 = elle répète les autres.
    explicatives = tableau.drop(columns=CIBLES).astype(float)
    avec_constante = np.column_stack([np.ones(len(explicatives)), explicatives.to_numpy()])
    with np.errstate(divide='ignore', invalid='ignore'):
        vif = [variance_inflation_factor(avec_constante, i + 1) for i in range(explicatives.shape[1])]
    resultat = pd.DataFrame({'Lien avec le salaire': explicatives.corrwith(tableau['Fixe_Annuel_MAD']),
                             'Lien avec le Compa-Ratio': explicatives.corrwith(tableau['Compa_Ratio']), 'VIF': vif})
    # Du lien le plus fort au plus faible avec le Compa-Ratio
    return resultat.sort_values('Lien avec le Compa-Ratio', key=abs, ascending=False)


display(liens_et_vif(chiffrees).round(2))

Le tableau aide à choisir les colonnes :

* **Lien avec le salaire** et **lien avec le Compa-Ratio** : plus il est loin de 0, plus la colonne compte pour cette cible.
* **VIF** : il dit si la colonne répète les autres colonnes explicatives. 1 = non ; au-dessus de 5 = à regarder ; au-dessus de 10 = oui.

**Comprendre le VIF.** Pour chaque colonne, on essaie de la deviner à partir de toutes les autres colonnes explicatives. Le R² de cet essai dit quelle part de la colonne on arrive à deviner, et :

$$\text{VIF} = \frac{1}{1 - R^2}$$

C'est comme une corrélation, mais « 1 contre toutes » au lieu de « 2 à 2 ».

Quand il n'y a que deux colonnes, les deux mesures disent la même chose : le VIF vaut 1 / (1 − r²), avec le r qu'on lit dans la matrice. Une corrélation de 0,9 entre deux colonnes donne un VIF de 5,3.

La différence apparaît dès qu'il y a plus de deux colonnes : le VIF peut être élevé alors qu'aucune case de la matrice ne l'est, parce que la colonne se devine avec une combinaison des autres, et pas avec une seule. Exemple : âge = âge à l'embauche + ancienneté.

On garde une colonne qui a un lien et un VIF bas. Une colonne au VIF élevé et au lien faible peut partir.

Une colonne en texte, comme le métier, n'est ni dans la matrice ni dans le tableau : les modèles ne l'utilisent pas.

Pour changer les colonnes : modifier la liste `COLONNES`, juste au-dessus de la matrice, puis relancer à partir de là.

**Ce que chaque colonne apporte.** Le tableau du dessus regarde les colonnes une par une. Ici on regarde le modèle entier : une régression apprend le Compa-Ratio avec toutes les colonnes chiffrées de la liste, puis sans une colonne, puis sans une autre.

À chaque fois, le modèle est jugé sur des salariés qu'il n'a pas vus : on coupe les salariés en 5 paquets, on apprend sur 4, on prédit le 5e, et on recommence pour chaque paquet.

* **Ce qu'elle apporte** : ce que le R² perd quand on retire la colonne, en %. Avec un R² de 0,45 qui tombe à 0,40 sans elle, elle apporte 5 %. Plus c'est grand, plus elle compte.
* Proche de 0 : le modèle fait aussi bien sans elle. Elle peut être mise de côté par un `#` dans la liste.

In [ ]:
from sklearn.model_selection import KFold

explicatives = chiffrees.drop(columns=CIBLES)
y_log = np.log(chiffrees['Compa_Ratio'])   # le Compa-Ratio en logarithme : quelques Compa-Ratio très hauts pèsent moins


def r2_sur_salaries_non_vus(colonnes):
    # Le R² moyen sur les 5 paquets : chaque paquet est prédit par un modèle appris sur les 4 autres
    r2 = []
    for apprend, verifie in KFold(5, shuffle=True, random_state=42).split(explicatives):
        modele = LinearRegression().fit(explicatives.iloc[apprend][colonnes], y_log.iloc[apprend])
        r2.append(r2_score(y_log.iloc[verifie], modele.predict(explicatives.iloc[verifie][colonnes])))
    return np.mean(r2)


toutes = list(explicatives.columns)
r2_toutes = r2_sur_salaries_non_vus(toutes)
print(f"R² du Compa-Ratio avec toutes les colonnes de la liste : {r2_toutes:.2f}")

# On retire chaque colonne à son tour
apport = pd.DataFrame({'R² sans elle': {c: r2_sur_salaries_non_vus([x for x in toutes if x != c]) for c in toutes}})
apport["Ce qu'elle apporte (%)"] = (r2_toutes - apport['R² sans elle']) * 100
display(apport.sort_values("Ce qu'elle apporte (%)", ascending=False).round(2))

**La décision.** Une colonne qui apporte beaucoup reste. Une colonne proche de 0 peut partir : on met un `#` devant elle dans la liste `COLONNES`, et on relance à partir de la liste.

Quand la liste convient, on continue.

---
## 5. Les six modèles

| Modèle | Explique | Avec |
|---|---|---|
| M1 | le salaire | une régression linéaire |
| M2 | le salaire | un arbre de décision |
| M3 | le Compa-Ratio | une régression linéaire |
| M4 | le Compa-Ratio | un arbre de décision |
| M5 | le salaire | un gradient boosting |
| M6 | le Compa-Ratio | un gradient boosting |

Un arbre de décision seul ne sait donner qu'un petit nombre de réponses différentes : avec une profondeur de 5, 32 au plus, et tous les salariés qui tombent dans la même case reçoivent la même valeur. Le gradient boosting additionne des centaines de petits arbres, chacun corrigeant les erreurs des précédents : il donne une valeur propre à chaque salarié, et il suit des effets qui ne sont pas en ligne droite (une ancienneté qui compte plus dans certains niveaux, par exemple).

80 % des salariés servent à apprendre (train), 20 % à vérifier (test) : les mêmes pour les six modèles. Les anomalies retirées à l'étape 3 ne sont ni dans le train ni dans le test.

Avec `LOG_SALAIRE = True`, le salaire est appris en logarithme, comme dans la régression du projet : un écart se lit alors en pourcentage (10 % de plus), et les très hauts salaires pèsent moins. Avec `False`, il est appris tel quel.

Avec `LOG_COMPA_RATIO = True`, le Compa-Ratio est lui aussi appris en logarithme, comme à l'étape 4 : quelques Compa-Ratio très hauts pèsent alors moins sur le modèle. Avec `False`, il est appris tel quel.

Les six modèles sont jugés sur le salaire, avec un seul résultat chacun :

* M1, M2 et M5 prédisent le salaire ; s'il est appris en logarithme, salaire prédit = exponentielle du log prédit ;
* M3, M4 et M6 prédisent le Compa-Ratio ; s'il est appris en logarithme, Compa-Ratio prédit = exponentielle du log prédit. Puis salaire prédit = Compa-Ratio prédit × `Mid`.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Les colonnes que les modèles reçoivent : celles choisies à l'étape 4, sans les deux cibles
X = donnees.drop(columns=CIBLES)
print(f"{X.shape[1]} colonnes pour {X.shape[0]} salariés : {list(X.columns)}")

# 1. On coupe en train et test, une seule fois. random_state : le même découpage à chaque lancement.
train, test = train_test_split(X.index, test_size=0.2, random_state=42)
print(f"Train : {len(train)} salariés. Test : {len(test)} salariés.")

# 2. Les six modèles : ce qu'ils expliquent, leur cible, et la façon de faire.
#    PROFONDEUR : le nombre de questions que l'arbre peut enchaîner. Sans limite, il apprend le train par cœur
#    et se trompe sur le test.
PROFONDEUR = 5
# LOG_SALAIRE. True : M1, M2 et M5 apprennent le logarithme du salaire, comme la régression du projet (un écart se lit
# alors en pourcentage, et les très hauts salaires pèsent moins). False : ils apprennent le salaire tel quel.
LOG_SALAIRE = True
# LOG_COMPA_RATIO. True : M3, M4 et M6 apprennent le logarithme du Compa-Ratio (quelques Compa-Ratio très hauts pèsent alors
# moins sur le modèle). False : ils apprennent le Compa-Ratio tel quel.
LOG_COMPA_RATIO = True
salaire = 'le salaire (en log)' if LOG_SALAIRE else 'le salaire'
compa_ratio = 'le Compa-Ratio (en log)' if LOG_COMPA_RATIO else 'le Compa-Ratio'
MODELES = {
    'M1': (f'{salaire}, par une régression', 'Fixe_Annuel_MAD', LinearRegression()),
    'M2': (f'{salaire}, par un arbre', 'Fixe_Annuel_MAD', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    'M3': (f'{compa_ratio}, par une régression', 'Compa_Ratio', LinearRegression()),
    'M4': (f'{compa_ratio}, par un arbre', 'Compa_Ratio', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    # M5 et M6 : un gradient boosting, des centaines de petits arbres qui se corrigent l'un l'autre. Pourquoi en plus de l'arbre
    # seul : un arbre de profondeur 5 ne sait donner que 32 valeurs différentes ; le gradient boosting en donne une par salarié,
    # et il suit des effets qui ne sont pas en ligne droite. random_state : le même résultat à chaque lancement.
    'M5': (f'{salaire}, par un gradient boosting', 'Fixe_Annuel_MAD', HistGradientBoostingRegressor(random_state=42)),
    'M6': (f'{compa_ratio}, par un gradient boosting', 'Compa_Ratio', HistGradientBoostingRegressor(random_state=42)),
}
# Les modèles du salaire reçoivent en plus le Mid : sans lui, ils ne connaissent pas la grille du poste.
# Les modèles du Compa-Ratio l'ont déjà, puisque le Compa-Ratio est le salaire divisé par le Mid.
# N'y mettre que des montants (Mid, Median...) : avec LOG_SALAIRE = True, ces colonnes passent elles aussi en logarithme.
# Liste vide : les six modèles ont exactement les mêmes colonnes.
EN_PLUS_POUR_LE_SALAIRE = ['Mid']

# 3. Chaque modèle apprend sur le train, puis prédit pour les salariés du test, qu'il n'a jamais vus.
#    Un seul résultat par modèle, toujours en salaire (MAD) :
#    - M1, M2 et M5 apprennent le salaire ; s'il est en logarithme, on revient au salaire par l'exponentielle ;
#    - M3, M4 et M6 apprennent le Compa-Ratio ; s'il est en logarithme, on revient au Compa-Ratio par l'exponentielle ;
#      salaire prédit = Compa-Ratio prédit × Mid.
#    L'erreur est le vrai moins le prédit.
mid = df.loc[test, 'Mid']
resultats = pd.DataFrame({'y_test_MAD': df.loc[test, 'Fixe_Annuel_MAD']})
for nom, (quoi, cible, modele) in MODELES.items():
    if cible == 'Fixe_Annuel_MAD':
        en_plus = [c for c in EN_PLUS_POUR_LE_SALAIRE if c in df.columns and c not in X.columns]
        if LOG_SALAIRE:
            X_modele = X.join(np.log(df[en_plus]).add_prefix('Log_'))
            modele.fit(X_modele.loc[train], np.log(df.loc[train, cible]))
            y_pred = np.exp(modele.predict(X_modele.loc[test]))
        else:
            X_modele = X.join(df[en_plus])
            modele.fit(X_modele.loc[train], df.loc[train, cible])
            y_pred = modele.predict(X_modele.loc[test])
    elif LOG_COMPA_RATIO:
        modele.fit(X.loc[train], np.log(df.loc[train, cible]))
        y_pred = np.exp(modele.predict(X.loc[test])) * mid
    else:
        modele.fit(X.loc[train], df.loc[train, cible])
        y_pred = modele.predict(X.loc[test]) * mid
    resultats[f'{nom}_y_pred_MAD'] = y_pred
    resultats[f'{nom}_erreur_MAD'] = resultats['y_test_MAD'] - resultats[f'{nom}_y_pred_MAD']
    # Un salaire prédit négatif ou nul est impossible : c'est le signe qu'une valeur aberrante est restée dans les données
    impossibles = int((resultats[f'{nom}_y_pred_MAD'] <= 0).sum())
    if impossibles:
        print(f"ATTENTION : {nom} prédit {impossibles} salaire(s) négatif(s) ou nul(s). Des valeurs aberrantes sont restées dans les données.")
display(resultats.head(10).round(2))

# 4. Les métriques des six modèles, sur le même test.
#    MAE et RMSE : sur le salaire, en MAD, pour les six.
#    R² : un seul par modèle, calculé sur ce que le modèle prédit. Le salaire pour M1, M2 et M5 ;
#    le Compa-Ratio pour M3, M4 et M6 (le salaire divisé par le Mid redonne le Compa-Ratio).
metriques = pd.DataFrame({
    f'{nom} : {quoi}': {
        'MAE (MAD)': mean_absolute_error(resultats['y_test_MAD'], resultats[f'{nom}_y_pred_MAD']),
        'RMSE (MAD)': mean_squared_error(resultats['y_test_MAD'], resultats[f'{nom}_y_pred_MAD']) ** 0.5,
        'R²': r2_score(resultats['y_test_MAD'], resultats[f'{nom}_y_pred_MAD']) if cible == 'Fixe_Annuel_MAD'
              else r2_score(resultats['y_test_MAD'] / mid, resultats[f'{nom}_y_pred_MAD'] / mid),
    } for nom, (quoi, cible, modele) in MODELES.items()}).T
display(metriques.round(2))

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : un seul par modèle, calculé sur ce que le modèle prédit. Pour M1, M2 et M5, c'est le salaire : il est haut dès que le salaire suit la grille, même si le modèle n'apprend rien d'autre. Pour M3, M4 et M6, c'est le Compa-Ratio : la part de l'écart autour du `Mid` que le modèle explique.

Le R² de M1, M2 et M5 ne se compare donc pas à celui de M3, M4 et M6. Pour comparer les six modèles, prendre la MAE et la RMSE : elles sont en MAD pour tous.

M5 et M6 répondent à une question : le salaire dépend-il des colonnes d'une façon qu'une ligne droite ne suit pas ? Si leur MAE est nettement plus basse que celle de M1 et de M3, oui. Si elle est presque la même, la limite vient des colonnes, pas du modèle.

In [ ]:
# Le salaire prédit face au vrai, pour les six modèles : un bon modèle met les points le long de la diagonale
points = pd.concat([pd.DataFrame({'Modèle': f'{nom} : {quoi}', 'y_test': resultats['y_test_MAD'],
                                  'y_pred': resultats[f'{nom}_y_pred_MAD'], 'erreur': resultats[f'{nom}_erreur_MAD']})
                    for nom, (quoi, cible, modele) in MODELES.items()])
fig = px.scatter(points, x='y_test', y='y_pred', facet_col='Modèle', facet_col_wrap=2, opacity=0.4, height=1300,
                 title='Salaire fixe prédit face au vrai (test)',
                 labels={'y_test': 'Vrai salaire fixe (y_test)', 'y_pred': 'Salaire fixe prédit (y_pred)'})
bornes = [points['y_test'].min(), points['y_test'].max()]
fig.add_scatter(x=bornes, y=bornes, mode='lines', line=dict(color='black'), showlegend=False, row='all', col='all')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

# Les erreurs sur le salaire : un bon modèle les serre autour de 0
fig = px.histogram(points, x='erreur', facet_col='Modèle', facet_col_wrap=2, nbins=60, height=1000,
                   title='Répartition des erreurs sur le salaire fixe (test)', labels={'erreur': 'Erreur (y_test − y_pred)'})
fig.add_vline(x=0, line_color='black')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

In [ ]:
# Export général en Excel : la base complète des salariés du test (toutes les colonnes du tableau, matricule compris),
# avec à côté le vrai salaire (y_test), puis le salaire prédit (y_pred) et l'erreur de chacun des six modèles.
export = df.loc[resultats.index].join(resultats).round(2)   # 2 chiffres après la virgule dans le fichier
# Le fichier s'écrit à côté du notebook
export.to_excel('resultats_prediction.xlsx', index=False)
print(f"{len(export)} salariés du test et {export.shape[1]} colonnes exportés dans resultats_prediction.xlsx")
export.head()

---
## 6. La validation croisée

À l'étape 5, les modèles sont jugés sur un seul test : 20 % des salariés, tirés une fois. Avec un autre tirage, les chiffres bougeraient un peu.

Ici, les salariés sont coupés en 5 paquets. Chaque paquet sert de test à son tour, pendant que les modèles apprennent sur les 4 autres. Chaque salarié est donc prédit une fois, par un modèle qui ne l'a pas vu.

Le reste ne change pas : les mêmes salariés, les mêmes six modèles, les mêmes réglages.

Lecture du tableau : chaque case est la moyenne des 5 paquets. C'est le chiffre à retenir, plus sûr que celui d'un seul test.

La validation croisée ne rend pas les modèles meilleurs. Elle donne des chiffres plus sûrs.

In [ ]:
from sklearn.model_selection import KFold

# 1. On coupe tous les salariés en paquets de même taille. random_state : les mêmes paquets à chaque lancement.
PAQUETS = 5
decoupage = KFold(n_splits=PAQUETS, shuffle=True, random_state=42)


def salaire_predit(modele, cible, apprend, verifie):
    # Le même calcul qu'à l'étape 5 : le modèle apprend sur les salariés « apprend »,
    # puis prédit le salaire (en MAD) des salariés « verifie ».
    # clone : un modèle neuf à chaque paquet, qui n'a rien gardé de l'apprentissage d'avant.
    modele = clone(modele)
    if cible == 'Fixe_Annuel_MAD':
        en_plus = [c for c in EN_PLUS_POUR_LE_SALAIRE if c in df.columns and c not in X.columns]
        if LOG_SALAIRE:
            X_modele = X.join(np.log(df[en_plus]).add_prefix('Log_'))
            modele.fit(X_modele.loc[apprend], np.log(df.loc[apprend, cible]))
            return np.exp(modele.predict(X_modele.loc[verifie]))
        X_modele = X.join(df[en_plus])
        modele.fit(X_modele.loc[apprend], df.loc[apprend, cible])
        return modele.predict(X_modele.loc[verifie])
    # Les modèles du Compa-Ratio : salaire prédit = Compa-Ratio prédit × Mid
    if LOG_COMPA_RATIO:
        modele.fit(X.loc[apprend], np.log(df.loc[apprend, cible]))
        return np.exp(modele.predict(X.loc[verifie])) * df.loc[verifie, 'Mid']
    modele.fit(X.loc[apprend], df.loc[apprend, cible])
    return modele.predict(X.loc[verifie]) * df.loc[verifie, 'Mid']


# 2. Chaque paquet sert de test à son tour, les autres paquets servent à apprendre
lignes = []
for numero, (apprend, verifie) in enumerate(decoupage.split(X), start=1):
    # KFold rend des positions (0, 1, 2...) : on retrouve les lignes du tableau, comme train et test à l'étape 5
    apprend, verifie = X.index[apprend], X.index[verifie]
    y_vrai, mid_paquet = df.loc[verifie, 'Fixe_Annuel_MAD'], df.loc[verifie, 'Mid']
    for nom, (quoi, cible, modele) in MODELES.items():
        y_pred = salaire_predit(modele, cible, apprend, verifie)
        # Les mêmes métriques qu'à l'étape 5, calculées sur ce paquet
        lignes.append({'Modèle': f'{nom} : {quoi}', 'Paquet': numero,
                       'MAE (MAD)': mean_absolute_error(y_vrai, y_pred),
                       'RMSE (MAD)': mean_squared_error(y_vrai, y_pred) ** 0.5,
                       'R²': r2_score(y_vrai, y_pred) if cible == 'Fixe_Annuel_MAD'
                             else r2_score(y_vrai / mid_paquet, y_pred / mid_paquet)})
# Une ligne par modèle et par paquet
par_paquet = pd.DataFrame(lignes)

# 3. Le même tableau qu'à l'étape 5, avec la moyenne des paquets dans chaque case
validation = par_paquet.groupby('Modèle', sort=False)[['MAE (MAD)', 'RMSE (MAD)', 'R²']].mean()
display(validation.round(2))